# Quantitative Trading Strategy Development  
## Step 06 — Strategy Backtesting

Objective:  
Simulate historical trading performance using a regime-aware strategy
and evaluate risk-adjusted returns.


## Pipeline Context

This notebook evaluates the trading strategy using historical data
and regime labels generated in previous steps.

Pipeline sequence:

1. Data Validation  
2. Data Cleaning  
3. Data Merging  
4. Feature Engineering  
5. Regime Detection  
6. Strategy Backtesting (current step)  
7. Machine Learning Enhancement  
8. Outlier Analysis  


In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

PLOTS_DIR = "../plots"
RESULTS_DIR = "../results"

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [2]:
from src.data_io import load_csv
from src.strategy import generate_signals, calculate_metrics
import joblib
import pandas as pd

## Loading Feature Dataset

The feature-engineered dataset is loaded as the input for
strategy simulation.


## Feature Selection

A subset of engineered features is selected to drive trading
signals and regime-aware decision logic.


In [4]:
df = load_csv('data/processed/nifty_features_5min.csv')

Loaded 126 rows from c:\Users\Tushar Mathur\Downloads\Quantitative_Trading_Strategy_Development\Quantitative_Trading_Strategy_Development\data/processed/nifty_features_5min.csv


## Strategy Design

The trading strategy follows a regime-aware framework:

- market regimes determine directional bias  
- signals are generated conditionally  
- positions are scaled based on volatility  
- execution avoids look-ahead bias  

This design allows dynamic adaptation to changing market conditions.


In [6]:
df = df.dropna()
feats = ['avg_iv', 'iv_spread', 'ema_5']
valid_feats = [f for f in feats if f in df.columns]
if not valid_feats: valid_feats = ['returns']

## Strategy Return Calculation

Strategy returns are computed by applying trading signals to market
returns with a one-period lag to prevent forward-looking bias.


In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

# 1. Prepare feature matrix
X = df[valid_feats].copy()
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()

# 2. Minimal data check
if X.shape[0] < 20:
    raise RuntimeError("Too few samples for regime detection")

# 3. Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Create and FIT HMM on the SAME data
model = GaussianHMM(
    n_components=3,
    covariance_type="diag",   # critical for stability
    n_iter=200,
    random_state=42
)

model.fit(X_scaled)

# 5. Predict regimes
regimes = model.predict(X_scaled)

# 6. Align back to dataframe
df_regime = df.loc[X.index].copy()
df_regime["regime"] = regimes

# 7. Validation output
print("Regime counts:")
print(df_regime["regime"].value_counts())


Regime counts:
regime
2    13
0     9
1     9
Name: count, dtype: int64


c:\Users\Tushar Mathur\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [8]:
df['regime'] = regimes

In [10]:
mapping = {0:0, 1:1, 2:-1} 
df['regime'] = df['regime'].map(mapping)

In [11]:
df['regime'] = df['regime'].map(mapping)

## Signal Generation

Trading signals are generated using predefined rules applied to
the selected features and detected market regime.


In [13]:
res = generate_signals(df)

## Performance Metrics

The following metrics are computed to evaluate strategy performance:

- Sharpe ratio  
- Win rate  
- Maximum drawdown  
- Calmar ratio  
- Total return  


In [15]:
metrics = calculate_metrics(res)
pd.DataFrame([metrics]).to_csv(f"{RESULTS_DIR}/strategy_metrics.csv", index=False)
print(metrics)

{'Sharpe': -22.09, 'Win Rate': 0.25, 'Max Drawdown': -0.01, 'Calmar': -0.67, 'Total Return': -0.01}


In [ ]:
import matplotlib.pyplot as plt

if 'strategy_returns' in res.columns:
    res['cumulative_returns'] = (1 + res['strategy_returns']).cumprod()
elif 'returns' in res.columns: 
    res['cumulative_returns'] = (1 + res['returns']).cumprod()

plt.figure(figsize=(10, 6))
if 'cumulative_returns' in res.columns:
    plt.plot(res['cumulative_returns'], label='Equity Curve')
    plt.title('Strategy Equity Curve')
    plt.xlabel('Time')
    plt.ylabel('Cumulative Returns')
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOTS_DIR}/equity_curve.png")
    plt.close()